<!-- VALIDMIND COPYRIGHT -->

<small>

***

Copyright © 2023-2026 ValidMind Inc. All rights reserved.<br>
Refer to [LICENSE](https://github.com/validmind/validmind-library/blob/main/LICENSE) for details.<br>
SPDX-License-Identifier: AGPL-3.0 AND ValidMind Commercial</small>

# Markdown and HTML logging regression (SC-17911)

This notebook verifies the compatibility contract for `log_text()` and `/tracking/log_metadata` after the WAF-safe TeX change. It covers plain Markdown, inline and display TeX, Markdown extensions, mixed Markdown/HTML, explicit HTML, and explicit MathJax HTML.

The first two test matrices are deterministic and do not require credentials or a running backend:

- When the backend advertises `log_metadata_markdown`, Markdown remains raw on the wire and carries `text_format="markdown"`. The SDK does not put generated MathJax `<script>` tags in the request.
- When the capability is absent, the SDK preserves compatibility with older backends by converting Markdown locally and omitting `text_format`.
- Explicit caller-supplied HTML remains unchanged in both modes. This intentionally includes explicit `<script>` input; the WAF-safe change only prevents the SDK from generating script tags from Markdown before transport.

The final section is an opt-in end-to-end check against a real backend. Set `VM_RUN_E2E=1` and the usual `VM_API_HOST`, `VM_API_KEY`, `VM_API_SECRET`, and `VM_API_MODEL` variables before executing the notebook. The E2E check creates or updates `test_description:sc17911_notebook_*` metadata records.

In [ ]:
import json
import os
from unittest.mock import patch

import pandas as pd
from bs4 import BeautifulSoup

import validmind as vm
import validmind.api_client as api_client
from validmind.client_config import client_config
from validmind.utils import is_html, md_to_html

CASES = [
    {
        "name": "plain_markdown",
        "text": "## Compatibility heading\n\nA **bold** paragraph and a list:\n\n- one\n- two",
    },
    {
        "name": "inline_tex",
        "text": r"Weight of Evidence is $WOE = \ln\dfrac{\%\ of\ Events}{\%\ of\ Non-Events}$.",
    },
    {
        "name": "display_tex",
        "text": "## Equation\n\n" + r"$$G(x) = \sum_{i=1}^{n} w_i x_i$$",
    },
    {
        "name": "markdown_extensions",
        "text": "~~Deprecated~~ current value.[^1]\n\n| Metric | Value |\n| --- | ---: |\n| AUC | 0.91 |\n\n[^1]: Regression footnote.",
    },
    {
        "name": "mixed_markdown_html",
        "text": "## Mixed input\n\nMarkdown before an <em>inline HTML fragment</em>.",
    },
    {
        "name": "explicit_html",
        "text": "<h2>Explicit HTML</h2><p><strong>Keep this markup unchanged.</strong></p>",
    },
    {
        "name": "explicit_mathjax_html",
        "text": r'<p><script type="math/tex">WOE = \ln\dfrac{\%\ of\ Events}{\%\ of\ Non-Events}</script></p>',
    },
]

assert {is_html(case["text"]) for case in CASES} == {False, True}
pd.DataFrame(
    {
        "case": [case["name"] for case in CASES],
        "detected_as": ["html" if is_html(case["text"]) else "markdown" for case in CASES],
    }
)

In [ ]:
async def capture_log_text_request(case, supports_markdown):
    """Run alog_text while capturing the serialized log_metadata body."""
    captured = {}

    async def capture_post(path, params=None, data=None, **kwargs):
        captured["path"] = path
        captured["params"] = params
        captured["body"] = json.loads(data)
        return {"content_id": captured["body"]["content_id"], "text": captured["body"].get("text")}

    original_flags = client_config.feature_flags
    try:
        client_config.feature_flags = {"log_metadata_markdown": supports_markdown}
        with patch.object(api_client, "_post", new=capture_post):
            await api_client.alog_text(
                content_id=f'test_description:sc17911_{case["name"]}',
                text=case["text"],
            )
    finally:
        client_config.feature_flags = original_flags

    assert captured["path"] == "log_metadata"
    return captured["body"]

## New backend: WAF-safe Markdown transport

Markdown and TeX must stay raw in the request. Fully formed HTML must retain the legacy pass-through behavior.

In [ ]:
new_backend_rows = []
for case in CASES:
    body = await capture_log_text_request(case, supports_markdown=True)
    html_input = is_html(case["text"])

    assert body["text"] == case["text"]
    if html_input:
        assert "text_format" not in body
    else:
        assert body["text_format"] == "markdown"
        assert "<script" not in body["text"].lower()

    new_backend_rows.append(
        {
            "case": case["name"],
            "text_format": body.get("text_format", "omitted"),
            "unchanged_on_wire": body["text"] == case["text"],
            "script_on_wire": "<script" in body["text"].lower(),
        }
    )

pd.DataFrame(new_backend_rows)

## Older backend: compatibility fallback

When the backend does not advertise support, Markdown must be converted locally and `text_format` must be omitted. Explicit HTML still passes through unchanged. This avoids silently storing raw Markdown on older customer-managed backends.

In [ ]:
legacy_backend_rows = []
for case in CASES:
    body = await capture_log_text_request(case, supports_markdown=False)
    html_input = is_html(case["text"])
    expected_text = case["text"] if html_input else md_to_html(case["text"], mathml=True)

    assert "text_format" not in body
    assert body["text"] == expected_text

    legacy_backend_rows.append(
        {
            "case": case["name"],
            "converted_locally": not html_input,
            "matches_legacy_output": body["text"] == expected_text,
            "script_on_wire": "<script" in body["text"].lower(),
        }
    )

pd.DataFrame(legacy_backend_rows)

## Optional end-to-end backend check

Run this section with `VM_RUN_E2E=1` to initialize the SDK from `VM_API_*`, log every case to the selected model's documentation, and verify the backend persists semantically equivalent rendered HTML while preserving explicit HTML byte-for-byte. The backend must advertise `feature_flags.log_metadata_markdown=true` for this check.

In [ ]:
run_e2e = os.getenv("VM_RUN_E2E", "0").lower() in {"1", "true", "yes"}
e2e_rows = []

def assert_backend_render(case, stored_text):
    """Assert stable rendering semantics without requiring byte-identical HTML."""
    if is_html(case["text"]):
        assert stored_text == case["text"]
        return

    assert stored_text != case["text"]
    soup = BeautifulSoup(stored_text, "html.parser")
    name = case["name"]

    if name == "plain_markdown":
        assert soup.find("h2").get_text(strip=True) == "Compatibility heading"
        assert soup.find("strong").get_text(strip=True) == "bold"
        assert [item.get_text(strip=True) for item in soup.find_all("li")] == ["one", "two"]
    elif name == "inline_tex":
        script = soup.find("script", attrs={"type": "math/tex"})
        assert script is not None and "WOE" in script.get_text()
    elif name == "display_tex":
        script = soup.find("script", attrs={"type": "math/tex"})
        assert script is not None and "block" in script.get("class", [])
        assert "G(x)" in script.get_text()
    elif name == "markdown_extensions":
        assert soup.find("del").get_text(strip=True) == "Deprecated"
        assert soup.select_one("figure.table table") is not None
        assert soup.select_one("section.footnotes") is not None
        assert [cell.get_text(strip=True) for cell in soup.find_all("th")] == ["Metric", "Value"]
    elif name == "mixed_markdown_html":
        assert soup.find("h2").get_text(strip=True) == "Mixed input"
        assert soup.find("em") is None
        assert "&lt;em&gt;inline HTML fragment&lt;/em&gt;" in stored_text
    else:
        raise AssertionError(f"Missing Markdown assertion for {name}")

if run_e2e:
    required = ["VM_API_HOST", "VM_API_KEY", "VM_API_SECRET", "VM_API_MODEL"]
    missing = [name for name in required if not os.getenv(name)]
    assert not missing, f"Missing required environment variables: {missing}"

    vm.init(document="documentation")
    assert client_config.supports_log_metadata_markdown(), (
        "The connected backend must advertise log_metadata_markdown=true "
        "for the end-to-end portion of this regression."
    )

    for case in CASES:
        response = await api_client.alog_text(
            content_id=f'test_description:sc17911_notebook_{case["name"]}',
            text=case["text"],
        )
        stored_text = response["text"]
        assert_backend_render(case, stored_text)

        e2e_rows.append(
            {
                "case": case["name"],
                "content_id": response["content_id"],
                "persisted_as_expected": True,
            }
        )

    await api_client._get_session().close()
else:
    e2e_rows.append(
        {
            "case": "skipped",
            "content_id": "Set VM_RUN_E2E=1 to run against a backend",
            "persisted_as_expected": None,
        }
    )

pd.DataFrame(e2e_rows)

## Headless execution

Run the deterministic transport regression from the repository root:

```bash
uv run jupyter nbconvert --execute --to notebook \
  --output /tmp/markdown_html_logging_regression.out.ipynb \
  notebooks/code_sharing/markdown_html_logging_regression.ipynb
```

Add `VM_RUN_E2E=1` and valid `VM_API_*` variables to the command environment to include the backend check. A failed assertion makes notebook execution exit non-zero.